# Federated Medical AI — Full Colab Run (Upload + Run All)

**Before running:** Runtime → Change runtime type → Hardware accelerator → **GPU** (T4 is fine, free tier).

**Then:** click **Runtime → Run all**. Cell 3 will pause and ask you to upload `federated-medical-ai.zip` - pick it when the button appears, everything else runs automatically after that.

This notebook deliberately does NOT reinstall numpy/torch/scipy - it only adds the packages that are missing, letting pip pick versions already compatible with what Colab ships. That's what avoids the numpy/faiss/sentence-transformers conflict from before - those errors only happen when numpy itself gets downgraded and re-upgraded repeatedly in the same session.

In [ ]:
!nvidia-smi

## 1. Upload the project zip
Click "Choose Files" below when it appears, and select `federated-medical-ai.zip` from your computer.

In [ ]:
from google.colab import files
import os

if not os.path.exists('/content/federated-medical-ai.zip'):
    uploaded = files.upload()
else:
    print('zip already present, skipping upload')

In [ ]:
%cd /content
!unzip -q -o federated-medical-ai.zip
%cd federated-medical-ai
!ls

## 2. Install only what's missing (no numpy/torch/scipy reinstall)

In [ ]:
!pip install -q faiss-cpu sentence-transformers langchain langchain-openai langgraph fastapi 'uvicorn[standard]' pydantic httpx ninja

In [ ]:
import torch, faiss, sentence_transformers, langchain, langgraph, fastapi

print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('faiss:', faiss.__version__)
print('sentence_transformers:', sentence_transformers.__version__)
print('fastapi:', fastapi.__version__)
assert torch.cuda.is_available(), 'GPU not available - check Runtime -> Change runtime type -> GPU'
print('\nAll imports OK, GPU available. Ready to continue.')

## 3. Build the custom CUDA quantization kernel
Compiles `cuda_kernels/quantize_kernel.cu` into a loadable PyTorch extension. Takes 1-2 minutes the first time.

In [ ]:
!rm -rf /root/.cache/torch_extensions

import sys
sys.path.insert(0, '.')

from torch.utils.cpp_extension import load

quant_cuda = load(
    name='quant_cuda',
    sources=['cuda_kernels/quantize_kernel.cu'],
    verbose=True,
)
sys.modules['quant_cuda'] = quant_cuda  # register globally so other modules' imports find it
print('Custom CUDA extension built and loaded successfully.')

## 4. Sanity check: quantize + dequantize round trip

In [ ]:
x = torch.randn(1000, device='cuda', dtype=torch.float32)
scale = (x.max() - x.min()).item() / 255.0
zero_point = round(-x.min().item() / scale)

q = quant_cuda.quantize_cuda(x, scale, zero_point)
dq = quant_cuda.dequantize_cuda(q, scale, zero_point, list(x.shape))

max_error = (x - dq).abs().max().item()
print(f'quantized dtype: {q.dtype}')
print(f'max reconstruction error: {max_error:.5f} (small error expected - lossy 8-bit quantization)')

## 5. Run federated training (synthetic placeholder dataset)
Swap the dataset in `data/dataset_loader.py` once the real one is ready - nothing else changes.

In [ ]:
from data.dataset_loader import get_dataset
from federated.train_federated import run_federated_training

dataset = get_dataset('synthetic')
train_ds, test_ds, info = dataset.load()

result = run_federated_training(
    train_ds, test_ds,
    num_classes=info.num_classes,
    input_channels=info.input_shape[0],
    num_clients=5,
    num_rounds=10,
    device='cuda',
)
print('Final accuracy:', result['history']['accuracy'][-1])

## 6. Benchmark: CPU vs naive-GPU vs custom-CUDA-kernel
This is the number for the README and the interview pitch.

In [ ]:
from cuda_kernels.benchmark import run_benchmark

results = run_benchmark(tensor_sizes=(10_000, 100_000, 1_000_000, 5_000_000), n_runs=20)

## 7. Plot the results

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 5))

ax[0].plot(results['tensor_size'], results['cpu_ms'], marker='o', label='CPU (PyTorch ops)')
ax[0].plot(results['tensor_size'], results['naive_gpu_ms'], marker='o', label='Naive GPU (PyTorch ops)')
ax[0].plot(results['tensor_size'], results['custom_cuda_ms'], marker='o', label='Custom CUDA kernel')
ax[0].set_xscale('log')
ax[0].set_yscale('log')
ax[0].set_xlabel('Tensor size (elements)')
ax[0].set_ylabel('Time per quantize+dequantize round trip (ms)')
ax[0].set_title('Quantization latency by implementation')
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].plot(results['tensor_size'], results['speedup_vs_cpu'], marker='o', label='Custom CUDA vs CPU')
ax[1].plot(results['tensor_size'], results['speedup_vs_naive_gpu'], marker='o', label='Custom CUDA vs naive GPU')
ax[1].set_xscale('log')
ax[1].set_xlabel('Tensor size (elements)')
ax[1].set_ylabel('Speedup (x)')
ax[1].set_title('Custom kernel speedup')
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('cuda_benchmark_results.png', dpi=150)
plt.show()
print('Saved plot to cuda_benchmark_results.png - download this from the file browser on the left, it is a great interview visual.')

## 8. (Optional) Federated RAG sanity check
Downloads a small embedding model the first time (needs internet, which Colab has).

In [ ]:
from rag.federated_retriever import FederatedRetriever
from rag.ingest import build_hospital_stores

stores = build_hospital_stores()
retriever = FederatedRetriever(stores)

result = retriever.retrieve('pituitary mass prolactin', top_k_final=5)
print('Hospitals queried:', result.hospitals_queried)
print('Possible conflict:', result.possible_conflict)
for s in result.snippets:
    print(f"  [{s.hospital_id}] score={s.score:.3f}  {s.text[:100]}...")